# 🏃‍♂️ Tugas Besar Kecerdasan Buatan (TE2514031) - Gasal 2026/2027
## Kelompok 13: Human Activity Recognition from Smartphone Sensors
### Milestone Minggu 12: Exploratory Data Analysis (EDA), Subject-Wise Split, & Baseline CNN-LSTM (PyTorch)

* **Dosen Pengampu**: Ir. Muhammad Ridho Dewanto, S.T., M.T.
* **Institusi**: Institut Teknologi Kalimantan (ITK)
* **Anggota Tim**:
  1. Muhammad Haekal
  2. Ali Rizqi Nurpebrianto
  3. Teodoro Junior Canoy
  4. Zuda Ersa Hidayat
* **Dataset**: UCI ID 240 - Human Activity Recognition Using Smartphones

---
Notebook ini dirancang agar dapat langsung dijalankan langkah demi langkah di **Google Colab** (dengan opsi akselerasi GPU: `Runtime -> Change runtime type -> T4 GPU`).


---
## Langkah 1: Setup Lingkungan dan Unduh Otomatis Dataset UCI HAR
Cell ini akan memeriksa dependensi dan mengunduh file dataset resmi UCI HAR langsung dari repositori publik UCI jika belum tersedia di folder `/content`.


In [ ]:
import os
import zipfile
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Cek Akselerasi Perangkat (GPU / CPU)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Device yang aktif: {device}")
if torch.cuda.is_available():
    print(f"   Nama GPU: {torch.cuda.get_device_name(0)}")

# Unduh dan Ekstrak Dataset UCI HAR jika belum ada
DATASET_DIR = "UCI HAR Dataset"

if not os.path.exists(DATASET_DIR):
    zip_url = "https://archive.ics.uci.edu/static/public/240/human+activity+recognition+using+smartphones.zip"
    zip_path = "har_dataset.zip"
    print("📥 Mengunduh dataset dari UCI Machine Learning Repository...")
    urllib.request.urlretrieve(zip_url, zip_path)
    
    print("📦 Mengekstrak berkas zip...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(".")
        
    # Ekstrak zip bersarang jika ada
    nested_zip = "UCI HAR Dataset.zip"
    if os.path.exists(nested_zip):
        with zipfile.ZipFile(nested_zip, 'r') as zip_ref:
            zip_ref.extractall(".")
            
    print("✅ Dataset berhasil disiapkan!")
else:
    print("✅ Folder dataset sudah tersedia di lingkungan Colab.")


---
## Langkah 2: Membaca Metadata dan Label Aktivitas
Membaca `activity_labels.txt` (6 kelas target) dan `features.txt` (561 nama variabel statistik sinyal sensor).


In [ ]:
# Load Metadata
activity_file = os.path.join(DATASET_DIR, "activity_labels.txt")
features_file = os.path.join(DATASET_DIR, "features.txt")

activity_labels = {}
with open(activity_file, "r") as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) >= 2:
            activity_labels[int(parts[0])] = parts[1]

feature_names = []
with open(features_file, "r") as f:
    for line in f:
        parts = line.strip().split(None, 1)
        if len(parts) >= 2:
            feature_names.append(parts[1])

print("🏷️ Daftar 6 Kelas Aktivitas:")
for k, v in activity_labels.items():
    print(f"   ID {k}: {v}")
print(f"
📊 Total Fitur Statistik Tersedia: {len(feature_names)} fitur")
print(f"   Contoh 5 fitur pertama: {feature_names[:5]}")


---
## Langkah 3: Membaca Dataset Fitur (X), Target (y), dan Identitas Subjek
Memuat `X_train.txt`, `y_train.txt`, `subject_train.txt`, `X_test.txt`, `y_test.txt`, dan `subject_test.txt`.


In [ ]:
print("⏳ Membaca data train dan test...")

X_train_full = pd.read_csv(os.path.join(DATASET_DIR, "train", "X_train.txt"), sep=r"\s+", header=None).values
y_train_full = pd.read_csv(os.path.join(DATASET_DIR, "train", "y_train.txt"), sep=r"\s+", header=None).values.squeeze()
sub_train_full = pd.read_csv(os.path.join(DATASET_DIR, "train", "subject_train.txt"), sep=r"\s+", header=None).values.squeeze()

X_test = pd.read_csv(os.path.join(DATASET_DIR, "test", "X_test.txt"), sep=r"\s+", header=None).values
y_test = pd.read_csv(os.path.join(DATASET_DIR, "test", "y_test.txt"), sep=r"\s+", header=None).values.squeeze()
sub_test = pd.read_csv(os.path.join(DATASET_DIR, "test", "subject_test.txt"), sep=r"\s+", header=None).values.squeeze()

print(f"✅ Data Train Shape: X={X_train_full.shape}, y={y_train_full.shape}, subjek={sub_train_full.shape}")
print(f"✅ Data Test Shape : X={X_test.shape}, y={y_test.shape}, subjek={sub_test.shape}")


---
## Langkah 4: Eksplorasi Data (EDA) & Validasi Integritas Bebas Leakage
1. Mengecek ketiadaan missing value (NaN/null).
2. Memverifikasi pembagian subjek untuk memastikan **tidak ada kebocoran data (*data leakage*)**.
3. Menganalisis dan memvisualisasikan distribusi sampel untuk setiap kelas aktivitas.


In [ ]:
# 1. Pengecekan Missing Value
nan_train = np.isnan(X_train_full).sum()
nan_test = np.isnan(X_test).sum()
print(f"🔍 Missing Values di Training Data : {nan_train}")
print(f"🔍 Missing Values di Testing Data  : {nan_test}")

# 2. Pengecekan Data Leakage Berdasarkan ID Partisipan
train_subs = np.unique(sub_train_full)
test_subs = np.unique(sub_test)
leakage = set(train_subs).intersection(set(test_subs))

print(f"
👥 Partisipan di Train Set ({len(train_subs)} orang): {train_subs}")
print(f"👥 Partisipan di Test Set  ({len(test_subs)} orang): {test_subs}")

if len(leakage) == 0:
    print("🛡️ [VERIFIKASI LOLOS]: Tidak ada data leakage! Subjek train dan test terpisah 100% secara ketat.")
else:
    print(f"⚠️ [PERINGATAN]: Terjadi data leakage pada subjek: {leakage}")

# 3. Analisis Distribusi Kelas Target
train_counts = pd.Series(y_train_full).value_counts().sort_index()
test_counts = pd.Series(y_test).value_counts().sort_index()

df_dist = pd.DataFrame({
    "Aktivitas": [activity_labels[i] for i in train_counts.index],
    "Jumlah Train": train_counts.values,
    "Persen Train (%)": (train_counts.values / len(y_train_full) * 100).round(2),
    "Jumlah Test": test_counts.values,
    "Persen Test (%)": (test_counts.values / len(y_test) * 100).round(2)
})
display(df_dist)

# Visualisasi Grafik Distribusi
fig, ax = plt.subplots(figsize=(10, 5))
x_idx = np.arange(len(df_dist))
width = 0.35
ax.bar(x_idx - width/2, df_dist["Jumlah Train"], width, label="Train Set (70% subjek)", color="#1F4E79")
ax.bar(x_idx + width/2, df_dist["Jumlah Test"], width, label="Test Set (30% subjek)", color="#E26D5C")
ax.set_xticks(x_idx)
ax.set_xticklabels(df_dist["Aktivitas"], rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Jumlah Sampel")
ax.set_title("Distribusi Kelas Aktivitas Manusia pada Dataset UCI HAR", fontsize=13, fontweight="bold")
ax.legend()
plt.tight_layout()
plt.show()


---
## Langkah 5: Partisi Data Berbasis Subjek (Subject-Wise Train/Validation Split)
Sesuai panduan mata kuliah, kita membagi 21 subjek di data training menjadi:
* **Train Set**: 16 subjek (5.459 sampel) untuk proses pelatihan model.
* **Validation Set**: 5 subjek (1.893 sampel) untuk evaluasi parameter dan tuning.
* **Test Set**: 9 subjek (2.947 sampel) tetap diisolasi secara ketat untuk pengujian akhir.


In [ ]:
# Tentukan 5 subjek untuk validation
val_subjects = [21, 27, 28, 29, 30]

train_mask = ~np.isin(sub_train_full, val_subjects)
val_mask = np.isin(sub_train_full, val_subjects)

X_train = X_train_full[train_mask]
y_train = y_train_full[train_mask] - 1 # Shift index 1..6 menjadi 0..5
sub_train = sub_train_full[train_mask]

X_val = X_train_full[val_mask]
y_val = y_train_full[val_mask] - 1
sub_val = sub_train_full[val_mask]

# Format Test Set
y_test_shifted = y_test - 1

print(f"📊 Pembagian Data Berbasis Subjek:")
print(f"   • Train Set : {X_train.shape[0]} sampel (Subjek: {np.unique(sub_train)})")
print(f"   • Val Set   : {X_val.shape[0]} sampel (Subjek: {np.unique(sub_val)})")
print(f"   • Test Set  : {X_test.shape[0]} sampel (Subjek: {np.unique(sub_test)})")


---
## Langkah 6: Pembuatan PyTorch DataLoader
Mengubah numpy array menjadi Tensor PyTorch dan membungkusnya ke dalam `DataLoader` dengan ukuran batch 64.


In [ ]:
batch_size = 64

train_dataset = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.long))
val_dataset = TensorDataset(torch.tensor(X_val, dtype=torch.float32), torch.tensor(y_val, dtype=torch.long))
test_dataset = TensorDataset(torch.tensor(X_test, dtype=torch.float32), torch.tensor(y_test_shifted, dtype=torch.long))

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

print("✅ DataLoaders siap digunakan untuk pelatihan.")


---
## Langkah 7: Arsitektur Model A (Baseline CNN-LSTM)
Membangun arsitektur hybrid **CNN-LSTM** menggunakan PyTorch:
1. **1D-CNN Block**: `Conv1d` (kernel=5, filter=64) + `BatchNorm1d` + `ReLU` + `MaxPool1d` untuk mengekstrak representasi fitur spasial lokal.
2. **LSTM Block**: `nn.LSTM` (hidden_size=64) untuk menangkap ketergantungan temporal urutan fitur.
3. **Classifier Head**: `Dropout` (0.3) + `Linear` output layer untuk 6 kelas aktivitas.


In [ ]:
class BaselineCNNLSTM(nn.Module):
    def __init__(self, num_features=561, num_classes=6, cnn_filters=64, lstm_units=64, dropout_rate=0.3):
        super(BaselineCNNLSTM, self).__init__()
        
        # 1D Convolutional Block
        self.conv1 = nn.Conv1d(in_channels=1, out_channels=cnn_filters, kernel_size=5, stride=1, padding=2)
        self.bn1 = nn.BatchNorm1d(cnn_filters)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool1d(kernel_size=2)
        
        # LSTM Block
        self.lstm = nn.LSTM(
            input_size=cnn_filters, 
            hidden_size=lstm_units, 
            num_layers=1, 
            batch_first=True
        )
        
        # Classifier
        self.dropout = nn.Dropout(dropout_rate)
        self.fc = nn.Linear(lstm_units, num_classes)

    def forward(self, x):
        # Input x: (batch_size, 561) -> Reshape ke (batch_size, 1, 561)
        x = x.unsqueeze(1)
        
        # CNN Feature Extraction
        x = self.conv1(x)
        x = self.bn1(x)
        x = self.relu(x)
        x = self.pool(x)
        
        # Permute ke format LSTM: (batch_size, seq_len, cnn_filters)
        x = x.permute(0, 2, 1)
        
        # LSTM Processing
        lstm_out, (hn, _) = self.lstm(x)
        
        # Ambil representasi hidden state terakhir
        last_hidden = hn[-1]
        
        # Dense Classifier
        out = self.dropout(last_hidden)
        logits = self.fc(out)
        return logits

# Inisialisasi Model A
num_features = X_train.shape[1]
num_classes = len(activity_labels)
model = BaselineCNNLSTM(num_features=num_features, num_classes=num_classes).to(device)

print(model)


---
## Langkah 8: Pelatihan Model A (Training Loop & Validasi)
Melatih model selama 12 epoch dengan loss function `CrossEntropyLoss` dan optimizer `Adam(lr=0.001)`. Model dengan skor `Validation Macro F1` tertinggi akan disimpan otomatis sebagai checkpoint terbaik.


In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss, correct, total = 0.0, 0, 0
    for inputs, labels in loader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * inputs.size(0)
        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)
    return running_loss / total, correct / total

def evaluate_model(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []
    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
    total = len(all_labels)
    acc = accuracy_score(all_labels, all_preds)
    prec, rec, f1, _ = precision_recall_fscore_support(all_labels, all_preds, average="macro", zero_division=0)
    return running_loss / total, acc, prec, rec, f1, all_preds, all_labels

# Konfigurasi Pelatihan
epochs = 12
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

history = {"train_loss": [], "val_loss": [], "train_acc": [], "val_acc": []}
best_val_f1 = 0.0
checkpoint_filename = "baseline_cnn_lstm_best.pt"

print("🚀 Memulai Pelatihan Model A (Baseline CNN-LSTM)...\n")
for epoch in range(1, epochs + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, vl_prec, vl_rec, vl_f1, _, _ = evaluate_model(model, val_loader, criterion)
    
    history["train_loss"].append(tr_loss)
    history["val_loss"].append(vl_loss)
    history["train_acc"].append(tr_acc)
    history["val_acc"].append(vl_acc)
    
    print(f"Epoch [{epoch:02d}/{epochs:02d}] "
          f"Train Loss: {tr_loss:.4f} | Train Acc: {tr_acc*100:.2f}% || "
          f"Val Loss: {vl_loss:.4f} | Val Acc: {vl_acc*100:.2f}% | Val F1: {vl_f1*100:.2f}%")
    
    if vl_f1 > best_val_f1:
        best_val_f1 = vl_f1
        torch.save(model.state_dict(), checkpoint_filename)

print(f"\n✅ Pelatihan Selesai! Checkpoint terbaik disimpan di '{checkpoint_filename}' (Val F1: {best_val_f1*100:.2f}%)")


---
## Langkah 9: Evaluasi Akhir pada Isolated Test Set (Bebas Data Leakage)
Memuat model terbaik dari checkpoint dan mengujinya pada Test Set yang tidak pernah dilihat model.


In [ ]:
# Muat bobot model terbaik
model.load_state_dict(torch.load(checkpoint_filename, map_location=device))

# Evaluasi pada Test Set
ts_loss, ts_acc, ts_prec, ts_rec, ts_f1, test_preds, test_labels = evaluate_model(model, test_loader, criterion)

print("=" * 60)
print("📊 HASIL EVALUASI MODEL A (TEST SET TERISOLASI BEBAS LEAKAGE)")
print("=" * 60)
print(f" Test Accuracy  : {ts_acc * 100:.2f}%")
print(f" Test Precision : {ts_prec * 100:.2f}%")
print(f" Test Recall    : {ts_rec * 100:.2f}%")
print(f" Test Macro F1  : {ts_f1 * 100:.2f}%")
print(f" Test Loss      : {ts_loss:.4f}")
print("=" * 60)


---
## Langkah 10: Visualisasi Confusion Matrix & Kurva Pembelajaran
Membuat visualisasi evaluasi lengkap untuk kebutuhan analisis laporan dan verifikasi dosen.


In [ ]:
# 1. Plot Confusion Matrix
activity_names = [activity_labels[i] for i in range(1, 7)]
cm = confusion_matrix(test_labels, test_preds)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)
ax.figure.colorbar(im, ax=ax)
ax.set(
    xticks=np.arange(cm.shape[1]),
    yticks=np.arange(cm.shape[0]),
    xticklabels=activity_names,
    yticklabels=activity_names,
    title="Confusion Matrix - Model A (Baseline CNN-LSTM)",
    ylabel="True Label (Aktual)",
    xlabel="Predicted Label (Prediksi)"
)
plt.setp(ax.get_xticklabels(), rotation=30, ha="right", rotation_mode="anchor")

thresh = cm.max() / 2.0
for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, format(cm[i, j], "d"),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black")
plt.tight_layout()
plt.show()

# 2. Plot Training & Validation Curves
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))

ax1.plot(history["train_loss"], label="Train Loss", color="#1F4E79", marker='o')
ax1.plot(history["val_loss"], label="Val Loss", color="#E26D5C", marker='s')
ax1.set_title("Kurva Loss (Model A)")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("CrossEntropy Loss")
ax1.legend()
ax1.grid(True, linestyle="--", alpha=0.6)

ax2.plot([a*100 for a in history["train_acc"]], label="Train Acc", color="#1F4E79", marker='o')
ax2.plot([a*100 for a in history["val_acc"]], label="Val Acc", color="#E26D5C", marker='s')
ax2.set_title("Kurva Akurasi (Model A)")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Akurasi (%)")
ax2.legend()
ax2.grid(True, linestyle="--", alpha=0.6)

plt.tight_layout()
plt.show()


---
## Langkah 11: Menyimpan dan Mengunduh Artefak untuk Streamlit
Menyimpan file hasil pelatihan agar dapat diunduh langsung dari Colab ke komputer lokal untuk digunakan pada dashboard Streamlit.


In [ ]:
import json

# Simpan ringkasan metrik ke JSON
metrics_dict = {
    "model_name": "Model A (Baseline CNN-LSTM)",
    "accuracy": float(ts_acc),
    "precision": float(ts_prec),
    "recall": float(ts_rec),
    "f1_score": float(ts_f1),
    "loss": float(ts_loss),
    "confusion_matrix": cm.tolist()
}

with open("baseline_metrics.json", "w") as f:
    json.dump(metrics_dict, f, indent=4)

print("✅ File 'baseline_metrics.json' dan 'baseline_cnn_lstm_best.pt' siap diunduh!")

# Uncomment baris di bawah jika ingin langsung mendownload file di Google Colab:
# from google.colab import files
# files.download('baseline_cnn_lstm_best.pt')
# files.download('baseline_metrics.json')
